# 데이터셋별 신경망 실습
## Iris · 작은 MLP

**구조:** 4 → 12 → 8 → 3 softmax  
**의도:** 작은 MLP로 표본이 적은 Iris의 과적합을 줄입니다.

### 학습 목표
- 입력 `X`와 정답 `y`의 의미 및 shape를 확인합니다.
- 훈련·검증·테스트를 나누고 훈련 기준으로 전처리합니다.
- 모델 정의, 학습 설정, 학습 실행을 순서대로 구현합니다.
- 예측을 평가하고 모델과 전처리를 저장·복원합니다.

### 실습 방법
1. 위에서부터 순서대로 실행하세요.
2. 각 절의 **확인할 내용**과 코드 주석을 함께 읽으세요.
3. 비교 실험은 설정 셀을 수정한 뒤 데이터 준비부터 다시 실행하세요.

그래프에는 영어를 사용합니다. 기본값은 전체 학습이며, `QUICK_RUN=True`는 2 epoch 점검 모드입니다.
구성은 비교 실험의 시작점입니다.

[셀 구성·주석 방식 참고: L03 Linear Regression](https://github.com/dearyonghoon/course-ai-programming-2026/blob/83c0f113a74b737f1790f3db9f41b34885fd57b2/labs/L03_Linear_Regression.ipynb)


## 0. 라이브러리 불러오기

표 데이터 처리, 그래프, 신경망 학습에 필요한 도구를 불러옵니다.

### 확인할 내용
- 패키지가 없을 때만 아래 설치 명령의 주석을 해제하세요.


In [ ]:
# Colab에서는 보통 별도 설치가 필요 없습니다. 로컬에서는 Python 3.11/3.12 권장.
# 아래 줄의 맨 앞 # 을 지운 뒤 실행하면 현재 노트북 커널에 설치됩니다.
# %pip install "tensorflow>=2.16,<3" "scikit-learn>=1.4,<2" "pandas>=2.1,<4" "numpy>=1.26,<3" "matplotlib>=3.8,<4" joblib


In [ ]:
import os
from pathlib import Path
import json
import platform
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 작은 소수와 큰 배열의 출력을 읽기 좋게 설정
np.set_printoptions(precision=4, suppress=True)


## 1. 실험 설정

자주 바꿀 값을 한 셀에 모았습니다. 설정을 바꾸면 이후 셀을 순서대로 다시 실행합니다.

### 확인할 내용
- 학습률·배치 크기·Dropout 비율을 확인하세요.
- 빠른 점검과 전체 학습을 구분하세요.


In [ ]:
from pathlib import Path

# 같은 분할과 초기 가중치로 실험을 비교하기 위한 난수 시드
SEED = 42
# 전체 실행에서 허용할 최대 반복 횟수. 조기 종료되면 더 적게 학습합니다.
MAX_EPOCHS = 300
# 한 번의 가중치 업데이트에 사용하는 샘플 수
BATCH_SIZE = 16
# 검증 손실이 개선되지 않아도 기다릴 epoch 수
PATIENCE = 15
# optimizer가 가중치를 한 번에 얼마나 바꿀지 결정하는 학습률
LEARNING_RATE = 0.001
# 학습 중 버릴 은닉층 출력의 비율. 0이면 비활성화됩니다.
DROPOUT_RATE = 0.0
# True는 파이프라인 확인용이며, 해당 결과를 최종 성능으로 해석하지 않습니다.
# True: 2 epoch로 실행 흐름만 확인. False: 전체 학습
QUICK_RUN = False
# 학습한 모델, 전처리, 평가 지표를 데이터셋별 폴더에 저장
OUTPUT_DIR = Path("artifacts") / "iris"


### 난수와 저장 폴더 준비

설정한 시드를 적용하고 실행 환경을 기록합니다.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VERSIONS = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn_version,
}
print(VERSIONS)


## 2. 데이터 살펴보기

입력은 꽃받침·꽃잎의 길이와 너비 4개, 정답은 붓꽃 품종 species입니다. Drive를 연결한 뒤 FILE_PATH를 확인하세요. 로컬에서는 이 값을 실제 CSV 경로로 수정합니다.

### 확인할 내용
- 첫 5행, 데이터 크기, 열 이름을 확인하세요.
- 어떤 열을 입력과 정답으로 사용하는지 확인하세요.


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Google Colab이 아닙니다. 로컬 경로를 FILE_PATH에 지정하세요.")

# Colab의 Drive 경로입니다. 로컬에서는 실제 CSV 경로로 수정하세요.
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/iris3.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 iris3.csv 위치로 수정하세요."
    )


In [ ]:
df = pd.read_csv(FILE_PATH)
# X는 모델 입력, y는 정답입니다. _raw는 전처리 전 값을 뜻합니다.
X_raw = df.drop(columns="species").astype("float32")
y_raw = df["species"]

assert X_raw.shape[1] == 4
print("Data:", df.shape)
print("Labels:", y_raw.value_counts().to_dict())


In [ ]:
print(df.head())
print("Rows / columns:", df.shape)


## 3. 훈련·검증·테스트 분할

로봇 데이터는 64/16/20, 다른 표 데이터는 60/20/20으로 분리합니다.

### 확인할 내용
- 훈련은 가중치 업데이트, 검증은 모델 선택, 테스트는 최종 평가에 사용합니다.


In [ ]:
X_tv_raw, X_test_raw, y_tv_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=SEED, stratify=y_raw
)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_tv_raw, y_tv_raw, test_size=0.25, random_state=SEED, stratify=y_tv_raw
)
print("Train / Validation / Test:", len(X_train_raw), len(X_val_raw), len(X_test_raw))


## 4. 전처리와 라벨 준비

훈련 기준으로 결측값을 대체하고 표준화합니다. 클래스 이름은 정수 라벨로 바꿉니다.

### 확인할 내용
- 변환 후 shape와 유한한 값인지 확인하세요.
- 새 샘플에도 저장된 전처리를 사용해야 합니다.


In [ ]:
# 결측값 대체와 스케일링을 하나로 묶어 추론 때 같은 순서로 재사용
preprocessor = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
# 훈련 데이터에서만 전처리 통계를 계산하고 변환
X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
# 검증·테스트는 훈련에서 구한 통계로 transform만 수행
X_val = preprocessor.transform(X_val_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")
# 문자 클래스 이름을 신경망이 사용할 정수 라벨로 변환
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw).astype("int32")
y_val = label_encoder.transform(y_val_raw).astype("int32")
y_test = label_encoder.transform(y_test_raw).astype("int32")
# 출력 확률의 열 수와 라벨 순서를 확인
N_CLASSES = len(label_encoder.classes_)
assert (
    np.isfinite(X_train).all()
    and np.isfinite(X_val).all()
    and np.isfinite(X_test).all()
)
print("Class order:", label_encoder.classes_)


> ### 체크포인트
> 검증·테스트를 이용해 전처리 통계를 다시 계산하지 않았는지 확인하세요.


## 5. 신경망 모델 정의

Input → 은닉층 → 출력층 순서로 네트워크를 구성합니다.

### 확인할 내용
- summary에서 각 층의 출력 shape와 파라미터 수를 확인하세요.


In [ ]:
hidden_units = (12, 8)
# 각 샘플의 특성 수를 입력 크기로 지정
model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        # ReLU 은닉층: 12개 뉴런, He 초기화
        Dense(12, activation="relu", kernel_initializer="he_normal"),
        # 학습 중에만 적용되며 predict에서는 자동으로 꺼집니다.
        Dropout(DROPOUT_RATE),
        # ReLU 은닉층: 8개 뉴런, He 초기화
        Dense(8, activation="relu", kernel_initializer="he_normal"),
        # 학습 중에만 적용되며 predict에서는 자동으로 꺼집니다.
        Dropout(DROPOUT_RATE),
        # 출력 수와 활성화는 문제의 정답 형태에 맞춥니다.
        Dense(N_CLASSES, activation="softmax"),
    ]
)
# 각 층의 출력 shape와 학습할 파라미터 수 확인
model.summary()


## 6. 손실 함수와 optimizer 설정

정수 클래스 라벨에는 softmax 출력과 sparse_categorical_crossentropy를 사용합니다. compile은 설정만 하며, 실제 가중치 갱신은 fit에서 시작됩니다.

### 확인할 내용
- 분류 출력과 cross entropy의 조합을 확인하세요.


In [ ]:
# optimizer, loss, 관찰할 metric을 설정합니다. 아직 학습하지 않습니다.
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

y_train_fit, y_val_fit = y_train, y_val


## 7. 모델 학습

검증 손실을 관찰하며 학습하고, 가장 좋았던 가중치를 복원합니다.

### 확인할 내용
- EarlyStopping의 patience와 restore_best_weights를 확인하세요.


In [ ]:
# 검증 손실이 정체되면 학습률을 줄이고, 더 개선되지 않으면 학습 종료
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
]

# 클래스 가중치를 사용하지 않는 기본값. 선택 옵션을 켜면 아래에서 계산
class_weight = None


### 학습 실행

batch_size는 업데이트당 샘플 수, epochs는 전체 데이터를 반복할 최대 횟수입니다.


In [ ]:
# fit은 반복 학습을 수행하고 epoch별 손실을 History에 기록
history = model.fit(
    X_train,
    y_train_fit,
    validation_data=(X_val, y_val_fit),
    epochs=2 if QUICK_RUN else MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=2,
)


## 8. 학습 곡선 확인

훈련 손실과 검증 손실을 같은 그래프에서 비교합니다.

### 확인할 내용
- 훈련 손실만 감소하고 검증 손실이 커지는 구간이 있는지 확인하세요.


In [ ]:
# loss는 훈련 손실, val_loss는 가중치 업데이트에 쓰지 않은 검증 손실입니다.
epochs_run = np.arange(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(7, 3))
plt.plot(epochs_run, history.history["loss"], label="Train")
plt.plot(epochs_run, history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=150)
plt.show()


> ### 체크포인트
> History에 실제로 학습한 epoch 수와 훈련·검증 손실이 기록되어 있는지 확인하세요.


## 9. 테스트 데이터로 평가

분류는 예측 라벨을 평가하고, 회귀는 예측을 원래 단위로 되돌린 뒤 평가합니다.

### 확인할 내용
- 기준선과 모델의 지표를 비교하세요.
- 테스트 결과로 설정을 다시 고르지 마세요.


In [ ]:
# 이진 분류에서는 이 확률 이상이면 양성 클래스로 판단
threshold = None
# 학습과 설정 선택을 마친 뒤 테스트 데이터의 예측 확률 계산
test_prob = model.predict(X_test, verbose=0)
# 분류에서는 예측 라벨, 회귀에서는 원래 단위의 예측값
pred = test_prob.argmax(axis=1)
metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro", zero_division=0)),
}

# 훈련에서 가장 흔한 클래스만 예측하는 단순 기준선과 비교
majority = int(np.bincount(y_train).argmax())
metrics["majority_baseline_accuracy"] = float(
    accuracy_score(y_test, np.full_like(y_test, majority))
)
print(json.dumps(metrics, indent=2))
print(
    classification_report(
        y_test,
        pred,
        labels=np.arange(N_CLASSES),
        target_names=[str(v) for v in label_encoder.classes_],
        zero_division=0,
    )
)


### 평가 결과 시각화

정답과 예측의 관계를 그래프로 확인합니다. 분류에서는 혼동행렬을 읽어 보세요.


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred), display_labels=label_encoder.classes_
).plot()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()


## 10. 모델과 전처리 저장

신경망뿐 아니라 전처리 통계와 입력 열의 순서도 함께 저장합니다.

### 확인할 내용
- .keras 파일과 preprocessing.joblib 파일이 만들어졌는지 확인하세요.


In [ ]:
def write_json(filename, value):
    """평가 결과와 설정을 UTF-8 JSON으로 저장합니다."""
    (OUTPUT_DIR / filename).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )


In [ ]:
# 학습한 가중치와 모델 구조를 하나의 .keras 파일에 저장
model.save(OUTPUT_DIR / "model.keras")
# 모델과 함께 전처리 통계·입출력 열 순서도 저장
joblib.dump(
    {
        "preprocessor": preprocessor,
        "label_encoder": label_encoder,
        "feature_columns": X_raw.columns.tolist(),
        "threshold": threshold,
    },
    OUTPUT_DIR / "preprocessing.joblib",
)


### 평가 기록 저장

손실 기록, 설정, 테스트 예측값을 파일로 남깁니다.


In [ ]:
write_json("metrics.json", metrics)
write_json(
    "config.json",
    {
        "seed": SEED,
        "hidden_units": list(hidden_units),
        "dropout_rate": DROPOUT_RATE,
        "learning_rate": LEARNING_RATE,
        "quick_run": QUICK_RUN,
        "threshold": threshold,
        "versions": VERSIONS,
    },
)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "history.csv", index=False)
pd.DataFrame(
    {
        "row_index": X_test_raw.index,
        "actual": label_encoder.inverse_transform(y_test),
        "predicted": label_encoder.inverse_transform(pred),
    }
).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## 11. 저장한 모델로 새 샘플 예측

같은 전처리를 적용한 뒤 모델을 호출하고, 필요한 역변환을 수행합니다.

### 확인할 내용
- 저장 전후의 예측이 같아야 합니다.
- 원시 입력을 scaler 없이 모델에 직접 넣지 마세요.


In [ ]:
# 저장한 신경망을 다시 불러와 새 입력을 예측
loaded_model = load_model(OUTPUT_DIR / "model.keras")
bundle = joblib.load(OUTPUT_DIR / "preprocessing.joblib")
# 샘플을 소수만 꺼내 저장된 전처리와 모델로 예측 과정을 확인
new_samples = X_test_raw.iloc[:3][bundle["feature_columns"]]
new_X = bundle["preprocessor"].transform(new_samples).astype("float32")
new_prob = loaded_model.predict(new_X, verbose=0)
# 저장 전후의 예측이 수치 오차 범위에서 같은지 확인
np.testing.assert_allclose(
    new_prob, model.predict(new_X, verbose=0), rtol=1e-5, atol=1e-6
)
new_class = (
    (new_prob.ravel() >= bundle["threshold"]).astype(int)
    if new_prob.shape[1] == 1
    else new_prob.argmax(axis=1)
)
print("Reloaded predictions:", bundle["label_encoder"].inverse_transform(new_class))
print("Saved to:", OUTPUT_DIR.resolve())


> ### 체크포인트
> 전처리 → 모델 예측 → 역변환 순서를 처음 학습할 때와 동일하게 유지했는지 확인하세요.


## 직접 해보기

**다음 비교:** Dropout·학습률만 바꾸어 validation을 비교하세요. test 결과를 보고 설정을 고르지 않습니다.
Iris의 Dropout 0.5가 항상 좋은 것은 아닙니다. 작은 test의 점수는 여러 seed에서 확인하세요.


## 정리

**데이터 확인 → 분할 → 훈련 기준 전처리 → 모델 정의 → compile → fit → 평가 → 저장·복원**

- `X_train`과 `y_train`으로 가중치를 학습합니다.
- 검증 데이터로 학습을 멈출 시점과 설정을 고릅니다.
- 테스트는 최종 평가에 사용합니다.
